# `unplug_charger` Transformer + Flow Matching

这份 notebook 是 AutoDL 自包含主训练入口。默认配置：
- 点云 + PointNet token
- DiT 风格 transformer backbone
- Flow Matching 策略
- `n_obs_steps = 3`
- 训练节奏、checkpoint、optimizer 尽量贴近 upstream transformer 配置


In [ ]:
from pathlib import Path
import sys

CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for cand in CANDIDATES:
    if (cand / 'lib').exists() and (cand / 'data').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate AutoDL project root.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.train_utils import ExperimentConfig, train_experiment, load_model_for_eval, run_success_rate_eval


In [ ]:
# User choices
RUN_NAME = 'unplug_charger_transformer_fm_obs3_dit_v1'
TRAIN_EPOCHS = 5000
BATCH_SIZE = 64
TRAIN_GRAD_ACCUM_STEPS = 2
NUM_WORKERS = 0

N_OBS_STEPS = 3
N_PRED_STEPS = 32
SUBS_FACTOR = 3
N_POINTS = 4096
USE_PC_COLOR = False
OBS_FEATURES_DIM = 256
Y_DIM = 10

HIDDEN_DIM = 512
TIME_DIM = 256
NUM_BLOCKS = 6
NHEAD = 8
DIM_FEEDFORWARD = 2048
DROPOUT = 0.1
ACTIVATION = 'gelu'

LEARNING_RATE = 1e-4
BETAS = (0.9, 0.95)
EPS = 1e-8
TRANSFORMER_WEIGHT_DECAY = 1e-3
OBS_ENCODER_WEIGHT_DECAY = 1e-6
LR_WARMUP_STEPS = 1000

EMA_ENABLE = True
EMA_DECAY = 0.9993
TRAIN_USE_AMP = False
GRAD_CLIP_NORM = 1.0

VAL_EVERY_EPOCHS = 1
SUCCESS_SELECTION_EVERY_EPOCHS = 50
CHECKPOINT_EVERY_EPOCHS = 50
SAMPLE_EVERY_EPOCHS = 5
SUCCESS_SELECTION_EPISODES = 10
SUCCESS_MAX_STEPS = 200
STANDARD_EVAL_EPISODES = 0

WANDB_ENABLE = True
WANDB_PROJECT = 'pfp-autodl-transformer-fm'
WANDB_ENTITY = None
WANDB_MODE = 'online'

# 24G fallback preset:
# BATCH_SIZE = 32
# TRAIN_GRAD_ACCUM_STEPS = 4


In [ ]:
cfg = ExperimentConfig(
    run_name=RUN_NAME,
    train_epochs=TRAIN_EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=TRAIN_GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    n_obs_steps=N_OBS_STEPS,
    n_pred_steps=N_PRED_STEPS,
    subs_factor=SUBS_FACTOR,
    n_points=N_POINTS,
    use_pc_color=USE_PC_COLOR,
    obs_features_dim=OBS_FEATURES_DIM,
    y_dim=Y_DIM,
    hidden_dim=HIDDEN_DIM,
    time_dim=TIME_DIM,
    num_blocks=NUM_BLOCKS,
    nhead=NHEAD,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    activation=ACTIVATION,
    learning_rate=LEARNING_RATE,
    betas=BETAS,
    eps=EPS,
    transformer_weight_decay=TRANSFORMER_WEIGHT_DECAY,
    obs_encoder_weight_decay=OBS_ENCODER_WEIGHT_DECAY,
    lr_warmup_steps=LR_WARMUP_STEPS,
    ema_enable=EMA_ENABLE,
    ema_decay=EMA_DECAY,
    train_use_amp=TRAIN_USE_AMP,
    grad_clip_norm=GRAD_CLIP_NORM,
    val_every_epochs=VAL_EVERY_EPOCHS,
    success_selection_every_epochs=SUCCESS_SELECTION_EVERY_EPOCHS,
    checkpoint_every_epochs=CHECKPOINT_EVERY_EPOCHS,
    sample_every_epochs=SAMPLE_EVERY_EPOCHS,
    success_selection_episodes=SUCCESS_SELECTION_EPISODES,
    success_max_steps=SUCCESS_MAX_STEPS,
    standard_eval_episodes=STANDARD_EVAL_EPISODES,
    wandb_enable=WANDB_ENABLE,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_mode=WANDB_MODE,
)
cfg.validate()
config_preview = {
    'run_name': cfg.run_name,
    'train_data_path': str(cfg.train_data_path),
    'valid_data_path': str(cfg.valid_data_path),
    'batch_size': cfg.batch_size,
    'grad_accum_steps': cfg.grad_accum_steps,
    'effective_batch_size': cfg.batch_size * cfg.grad_accum_steps,
    'train_epochs': cfg.train_epochs,
    'n_obs_steps': cfg.n_obs_steps,
    'hidden_dim': cfg.hidden_dim,
    'num_blocks': cfg.num_blocks,
    'nhead': cfg.nhead,
    'checkpoint_every_epochs': cfg.checkpoint_every_epochs,
    'success_selection_every_epochs': cfg.success_selection_every_epochs,
    'device': str((PROJECT_ROOT / 'lib').exists()),
}
config_preview


In [ ]:
summary = train_experiment(cfg, strategy='fm')
summary


In [ ]:
# Optional: manual standard eval on a chosen checkpoint
RUN_STANDARD_EVAL = False
EVAL_CKPT = cfg.latest_ckpt_path
EVAL_EPISODES = 50

if RUN_STANDARD_EVAL:
    model, payload = load_model_for_eval(cfg, strategy='fm', ckpt_path=EVAL_CKPT, prefer_ema=True)
    standard_summary = run_success_rate_eval(
        model,
        cfg,
        num_episodes=EVAL_EPISODES,
        max_steps=cfg.success_max_steps,
        headless=True,
        show_progress=True,
        progress_desc='standard eval',
    )
    standard_summary
